<a href="https://colab.research.google.com/github/Vinod1204/Machine-Learning/blob/Machine-Learning/Linear_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import

In [4]:
import math
import numpy as np
import pandas as pd
import plotly.express as px
import pickle
from google.colab import files

Data Loading and Analysis

In [9]:
uploaded=files.upload()
# Load the training and test datasets
train_data =pd.read_csv("train.csv")

# Remove rows with missing values
train_data = train_data.dropna()


Saving train.csv to train (3).csv


In [11]:
uploaded=files.upload()
test_data = pd.read_csv('test.csv')
test_data = test_data.dropna()

Saving test.csv to test.csv


In [13]:
train_data.head()

,x,y
0,24.0,21.549452
1,50.0,47.464463
2,15.0,17.218656
3,38.0,36.586398
4,87.0,87.288984


In [14]:
px.scatter(x=train_data['x'], y=train_data['y'],template='seaborn')

Data Preprocessing

In [15]:
# Set training data and target
X_train = train_data['x'].values
y_train = train_data['y'].values

# Set testing data and target
X_test = test_data['x'].values
y_test = test_data['y'].values

In [16]:
def standardize_data(X_train, X_test):
    """
    Standardizes the input data using mean and standard deviation.

    Parameters:
        X_train (numpy.ndarray): Training data.
        X_test (numpy.ndarray): Testing data.

    Returns:
        Tuple of standardized training and testing data.
    """
    # Calculate the mean and standard deviation using the training data
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)

    # Standardize the data
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    return X_train, X_test

X_train, X_test = standardize_data(X_train, X_test)

Reshaping data

In [17]:
X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

Model Implementation

In [21]:
class LinearRegression:


    def __init__(self, learning_rate, convergence_tol=1e-6):
        self.learning_rate = learning_rate
        self.convergence_tol = convergence_tol
        self.W = None
        self.b = None

    def initialize_parameters(self, n_features):

        self.W = np.random.randn(n_features) * 0.01
        self.b = 0

    def forward(self, X):

        return np.dot(X, self.W) + self.b

    def compute_cost(self, predictions):

        m = len(predictions)
        cost = np.sum(np.square(predictions - self.y)) / (2 * m)
        return cost

    def backward(self, predictions):

        m = len(predictions)
        self.dW = np.dot(predictions - self.y, self.X) / m
        self.db = np.sum(predictions - self.y) / m

    def fit(self, X, y, iterations, plot_cost=True):

        assert isinstance(X, np.ndarray), "X must be a NumPy array"
        assert isinstance(y, np.ndarray), "y must be a NumPy array"
        assert X.shape[0] == y.shape[0], "X and y must have the same number of samples"
        assert iterations > 0, "Iterations must be greater than 0"

        self.X = X
        self.y = y
        self.initialize_parameters(X.shape[1])
        costs = []

        for i in range(iterations):
            predictions = self.forward(X)
            cost = self.compute_cost(predictions)
            self.backward(predictions)
            self.W -= self.learning_rate * self.dW
            self.b -= self.learning_rate * self.db
            costs.append(cost)

            if i % 100 == 0:
                print(f'Iteration: {i}, Cost: {cost}')

            if i > 0 and abs(costs[-1] - costs[-2]) < self.convergence_tol:
                print(f'Converged after {i} iterations.')
                break

        if plot_cost:
            fig = px.line(y=costs, title="Cost vs Iteration", template="plotly_dark")
            fig.update_layout(
                title_font_color="#41BEE9",
                xaxis=dict(color="#41BEE9", title="Iterations"),
                yaxis=dict(color="#41BEE9", title="Cost")
            )

            fig.show()

    def predict(self, X):

        return self.forward(X)


    def save_model(self, filename=None):

        model_data = {
            'learning_rate': self.learning_rate,
            'convergence_tol': self.convergence_tol,
            'W': self.W,
            'b': self.b
        }

        with open(filename, 'wb') as file:
            pickle.dump(model_data, file)

    @classmethod
    def load_model(cls, filename):

        with open(filename, 'rb') as file:
            model_data = pickle.load(file)

        # Create a new instance of the class and initialize it with the loaded parameters
        loaded_model = cls(model_data['learning_rate'], model_data['convergence_tol'])
        loaded_model.W = model_data['W']
        loaded_model.b = model_data['b']

        return loaded_model

In [22]:
lr = LinearRegression(0.01)
lr.fit(X_train, y_train, 10000)

Iteration: 0, Cost: 1670.3564819191258
Iteration: 100, Cost: 227.20063523460965
Iteration: 200, Cost: 33.84708412645159
Iteration: 300, Cost: 7.94163821634818
Iteration: 400, Cost: 4.470834996261984
Iteration: 500, Cost: 4.00581790933886
Iteration: 600, Cost: 3.943515071229502
Iteration: 700, Cost: 3.9351677572368824
Iteration: 800, Cost: 3.934049386822216
Converged after 863 iterations.


In [20]:
lr.save_model('model.pkl')

In [23]:
model = LinearRegression.load_model("model.pkl")

Evaluation

In [24]:
class RegressionMetrics:
    @staticmethod
    def mean_squared_error(y_true, y_pred):
        assert len(y_true) == len(y_pred), "Input arrays must have the same length."
        mse = np.mean((y_true - y_pred) ** 2)
        return mse

    @staticmethod
    def root_mean_squared_error(y_true, y_pred):
        assert len(y_true) == len(y_pred), "Input arrays must have the same length."
        mse = RegressionMetrics.mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        return rmse

    @staticmethod
    def r_squared(y_true, y_pred):

        assert len(y_true) == len(y_pred), "Input arrays must have the same length."
        mean_y = np.mean(y_true)
        ss_total = np.sum((y_true - mean_y) ** 2)
        ss_residual = np.sum((y_true - y_pred) ** 2)
        r2 = 1 - (ss_residual / ss_total)
        return r2

In [25]:
y_pred = model.predict(X_test)
mse_value = RegressionMetrics.mean_squared_error(y_test, y_pred)
rmse_value = RegressionMetrics.root_mean_squared_error(y_test, y_pred)
r_squared_value = RegressionMetrics.r_squared(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse_value}")
print(f"Root Mean Squared Error (RMSE): {rmse_value}")
print(f"R-squared (Coefficient of Determination): {r_squared_value}")

Mean Squared Error (MSE): 9.442669814089326
Root Mean Squared Error (RMSE): 3.0728927436683056
R-squared (Coefficient of Determination): 0.9887898722725122


In [27]:
# Create a DataFrame for actual vs predicted values
results_df = pd.DataFrame({'Actual': y_test.flatten(), 'Predicted': y_pred.flatten()})
print("Actual vs Predicted Values (First 15 Samples):")
display(results_df.head(15))

# Create a DataFrame for metrics
metrics_data = {'Metric': ['Mean Squared Error (MSE)', 'Root Mean Squared Error (RMSE)', 'R-squared (Coefficient of Determination)'],
                'Value': [mse_value, rmse_value, r_squared_value]}
metrics_df = pd.DataFrame(metrics_data)
print("\nRegression Metrics:")
display(metrics_df)

Actual vs Predicted Values (First 15 Samples):


,Actual,Predicted
0,79.775152,76.930245
1,23.177279,20.902978
2,25.609262,21.903465
3,17.857388,19.902491
4,41.849864,35.910282
5,9.805235,14.900056
6,58.874659,61.922941
7,97.617937,94.939009
8,18.395127,19.902491
9,8.746748,4.895187



Regression Metrics:


,Metric,Value
0,Mean Squared Error (MSE),9.442670
1,Root Mean Squared Error (RMSE),3.072893
2,R-squared (Coefficient of Determination),0.988790
